<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/dynatree-cre/notebooks/m7_2_m7_3_colab_seed2021.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M7.2 & M7.3 — Dual-Space, Size-Corrected & Hybrid Routing Benchmark (Colab GPU, Seed 2021)
- **M7.2 (Dual-Space):** Tách `similarity_space` (xây cây) và `retrieval_similarity_space` (truy xuất ứng viên), đối chứng 2×2.
- **M7.3 (Cluster-Size Bias & Hybrid):** `size_corrected_hierarchical` (`eta=1.0`) và `hybrid_tree_routing` (`gamma=0.25` vs control `gamma=0.0`) trên FewRel Standard, Exp B (`semantic_homogeneous`), Exp C (`cross_task_dispersed`), và TACRED.
- **M7.1 Auto-Logging:** Mọi lượt chạy tự động ghi kèm `2×2 decomposition`, `Recall@1/3/5/10`, `per_cluster`, `zero_grad_new_leaf_count` và backup `.pt` lên Google Drive.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Chon GPU runtime (T4/L4/A100)!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/vinhlq2512/research-code-lab.git'
BRANCH = 'feat/dynatree-cre'
REPO_DIR = '/content/research-code-lab'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scipy scikit-learn tabulate pytest
!python scripts/setup_protocol_v1_data.py

gdrive_runs = Path('/content/drive/MyDrive/TAPTA_DynaTree_CRE/runs/dynatree_cre')
gdrive_runs.mkdir(parents=True, exist_ok=True)
local_runs = Path('/content/research-code-lab/runs/dynatree_cre')
local_runs.parent.mkdir(parents=True, exist_ok=True)
if local_runs.is_symlink():
    local_runs.unlink()
elif local_runs.exists():
    shutil.copytree(local_runs, gdrive_runs, dirs_exist_ok=True)
    shutil.rmtree(local_runs)
local_runs.symlink_to(gdrive_runs)
!pytest tests/dynatree_cre -q
print('Setup & Drive backup ready ->', gdrive_runs)

In [ ]:
# Part A — M7.2 Dual-Space 2x2 Matrix (FewRel Standard, Seed 2021)
# (Note: M7.2-B Centered+Centered = DT-C4 MVP da chay 57.23% NCM)

# M7.2-A: Raw Tree (0.75) + Raw Retrieval
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_2_dual_space_centered_tree_raw_route_fewrel_5shot_seed2021.yaml \
    --similarity-space raw_cosine --retrieval-similarity-space raw_cosine \
    --threshold-mode fixed --insertion-threshold 0.75 \
    --output-dir runs/dynatree_cre/m7_2_raw_tree_raw_route/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# M7.2-C: Centered Tree (q=0.70) + Raw Retrieval (De xuat chinh)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_2_dual_space_centered_tree_raw_route_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

# M7.2-D: Raw Tree (0.75) + Centered Retrieval
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_2_dual_space_centered_tree_raw_route_fewrel_5shot_seed2021.yaml \
    --similarity-space raw_cosine --retrieval-similarity-space t1_centered_cosine \
    --threshold-mode fixed --insertion-threshold 0.75 \
    --output-dir runs/dynatree_cre/m7_2_raw_tree_centered_route/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

In [ ]:
# Part B — M7.3 Size-Corrected & Hybrid Routing (FewRel Standard, Exp B, Exp C, TACRED)

# 1. FewRel Standard: Size-Corrected (eta=1.0), Hybrid (gamma=0.25), Hybrid Control (gamma=0.0 = Flat Raw)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_size_corrected_dual_space_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_fewrel_5shot_seed2021.yaml \
    --device cuda --allow-hub-download
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_fewrel_5shot_seed2021.yaml \
    --hybrid-cluster-gamma 0.0 \
    --output-dir runs/dynatree_cre/m7_3_hybrid_gamma00_flat/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# 2. Exp B (semantic_homogeneous) — noi DT-C4 bi cluster-size bias nang nhat
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_size_corrected_dual_space_fewrel_5shot_seed2021.yaml \
    --task-distribution semantic_homogeneous \
    --output-dir runs/dynatree_cre/m7_3_size_corrected_eta1_expB/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_fewrel_5shot_seed2021.yaml \
    --task-distribution semantic_homogeneous \
    --output-dir runs/dynatree_cre/m7_3_hybrid_gamma025_expB/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_fewrel_5shot_seed2021.yaml \
    --task-distribution semantic_homogeneous --hybrid-cluster-gamma 0.0 \
    --output-dir runs/dynatree_cre/m7_3_hybrid_gamma00_expB/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# 3. Exp C (cross_task_dispersed)
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_fewrel_5shot_seed2021.yaml \
    --task-distribution cross_task_dispersed \
    --output-dir runs/dynatree_cre/m7_3_hybrid_gamma025_expC/fewrel_5shot/seed_2021 \
    --device cuda --allow-hub-download

# 4. TACRED: Size-Corrected & Hybrid
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_tacred_5shot_seed2021.yaml \
    --routing-strategy size_corrected_hierarchical --cluster-size-eta 1.0 \
    --output-dir runs/dynatree_cre/m7_3_size_corrected_eta1/tacred_5shot/seed_2021 \
    --device cuda --allow-hub-download
!python experiments/run_dynatree_cre.py \
    --config configs/dynatree_cre/m7_3_hybrid_dual_space_tacred_5shot_seed2021.yaml \
    --device cuda --allow-hub-download

In [ ]:
# Part C — Summary Table & Export Bundle (khong kem *.pt)
import json
import os
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

runs_root = Path('/content/research-code-lab/runs/dynatree_cre')
rows = []
bundle = Path('/content/m7_2_m7_3_bundle')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

for m_path in sorted(runs_root.glob('m7_*/**/metrics.json')):
    run_dir = m_path.parent
    rel = run_dir.relative_to(runs_root)
    shutil.copytree(run_dir, bundle / rel, ignore=shutil.ignore_patterns('*.pt'), dirs_exist_ok=True)
    d = json.loads(m_path.read_text())
    r = json.loads((run_dir / 'routing_metrics.json').read_text())
    last_r = r['stages'][-1]
    eval_r = last_r.get('eval_routing', {})
    decomp = last_r.get('m7_1_decomposition', {})
    rk = decomp.get('recall_at_k', {})
    cond = decomp.get('conditional_ncm_acc_given_in_topk', {})
    rows.append([
        rel.parts[0],
        d.get('dataset'),
        d.get('task_distribution'),
        f"{d.get('final_ncm_whole_accuracy', 0)*100:.2f}%",
        f"{d.get('ncm_summary', {}).get('average_forgetting', 0)*100:.2f}%",
        f"{eval_r.get('relation_routing_top1_acc', 0)*100:.2f}%",
        f"{rk.get('R@3', 0)*100:.2f}%",
        f"{(cond.get('R@3') or 0)*100:.2f}%",
        f"{eval_r.get('cluster_routing_top1_acc', 0)*100:.2f}%",
        d.get('final_tree_metrics', {}).get('num_clusters'),
    ])

headers = ['Run', 'Dataset', 'Dist', 'NCM Acc', 'NCM F', 'Route@1', 'R@3', 'Cond@3', 'Clus@1', 'Clusters']
print(tabulate(rows, headers=headers, tablefmt='grid'))

shutil.make_archive('/content/m7_2_m7_3_colab_results', 'zip', bundle)
print('Bundle size KB:', os.path.getsize('/content/m7_2_m7_3_colab_results.zip') / 1024)
files.download('/content/m7_2_m7_3_colab_results.zip')